In [ ]:
import sys
print(sys.executable)

import pandas, sklearn
print(pandas.__version__, sklearn.__version__)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
df.shape

df.columns.to_list()


In [ ]:
df['Class'].value_counts(normalize=True)

#df.info()

In [ ]:
df.duplicated().sum()


In [ ]:
df.describe()

In [ ]:
print(df['Time'].head())

print(df['Time'].min()) 
print(df['Time'].max())


In [ ]:
df['hour_of_day'] = (df['Time'] % 86400) // 3600
print(df[['Time', 'hour_of_day']].head())

In [ ]:
print(df['hour_of_day'].unique())

df.groupby('hour_of_day')['Class'].mean()

In [ ]:
df.groupby('hour_of_day')['Class'].sum()

In [ ]:
#all dupicate rows 
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10) 

In [ ]:
df = df.drop_duplicates()
df.shape

In [ ]:
df.groupby('Class')['Amount'].describe()

In [ ]:
sns.boxplot(x='Class', y='Amount', data=df[df['Amount'] < 500])
plt.show()

In [ ]:
# only catches linear relationships
correlations = df.corr()['Class'].sort_values(ascending=False)
correlations

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)

In [ ]:
X_train.filter(like='V').skew()

In [ ]:
from sklearn.preprocessing import RobustScaler

cols_to_scale = X_train.columns.to_list()
scaler = RobustScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

X_train_scaled.describe()

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)

print(classification_report(y_test, y_pred_dummy))

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
logreg.fit(X_train_scaled, y_train)
y_pred_logreg = logreg.predict(X_test_scaled)

print(classification_report(y_test, y_pred_logreg))